In [1]:
# ============================================================
# RAM-OPTIMIZED + CHECKPOINT/RESUME + TRAIN/VAL/TEST REPORT
# CNNs + Weighted Soft-Voting + Stacking + Focused Grad-CAM++
# OPTIMIZED FOR KAGGLE TESLA P100 GPU
# ============================================================

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import gc, math, json, time, shutil, random, warnings
from datetime import datetime
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import psutil
import joblib
import kagglehub

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import StratifiedKFold
import sklearn.metrics as skm
from sklearn.preprocessing import label_binarize
from sklearn.linear_model import LogisticRegression

try:
    from statsmodels.stats.contingency_tables import mcnemar
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    from statsmodels.stats.contingency_tables import mcnemar

from tensorflow.keras.applications import EfficientNetB0, EfficientNetB4, MobileNetV2, MobileNetV3Large, ResNet50V2, VGG16
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mnv2
from tensorflow.keras.applications.resnet_v2 import preprocess_input as preprocess_resnetv2
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED); tf.keras.utils.set_random_seed(SEED)

MODELS_TO_RUN = ['mobilenetv2','mobilenetv3','resnet50v2','efficientnetb0','efficientnetb4','vgg16']
IMG_SIZE_DICT = {m:(224,224) for m in MODELS_TO_RUN}
BATCH_SIZE_DICT = {'mobilenetv2':16,'mobilenetv3':16,'resnet50v2':16,'efficientnetb0':16,'vgg16':16,'efficientnetb4':8}

RUN_KFOLD = True
RUN_FINAL_TRAINING = True
RUN_WEIGHTED_ENSEMBLE = True
RUN_STACKING_ENSEMBLE = True
RUN_GRADCAM = True
RUN_ENSEMBLE_GRADCAM = True
FAST_DEBUG = False

N_SPLITS = 5
SELECTIVE_TOP_K_LIST = [2,3,4]
HEAD_EPOCHS = 8 if not FAST_DEBUG else 1
FINE_TUNE_EPOCHS = 10 if not FAST_DEBUG else 1
EARLY_STOPPING_PATIENCE = 5 if not FAST_DEBUG else 1

LR_HEAD = 1e-3
LR_FINE = 1e-5
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
DROPOUT_RATE = 0.4
DENSE_UNITS = 256

USE_ONLINE_AUGMENTATION = True
USE_LABEL_SMOOTHING = True
LABEL_SMOOTHING_VALUE = 0.1
MENINGIOMA_OVERSAMPLE_FACTOR = 0.5
MENINGIOMA_CLASS_IDX = 1

GRADCAM_N_IMAGES = 8
GRADCAM_THRESHOLD_PERCENTILE = 85
GRADCAM_ALPHA = 0.30
USE_GRADCAM_PLUS_PLUS = True

FORCE_RETRAIN = False
FORCE_REEVALUATE = False
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
AUTOTUNE = tf.data.AUTOTUNE

# ============================================================
# ENVIRONMENT & HARDWARE OPTIMIZATION (P100 SPECIFIC)
# ============================================================

def print_ram(tag='RAM'):
    try:
        r = psutil.virtual_memory(); print(f'{tag}: used={r.used/1e9:.2f}GB | available={r.available/1e9:.2f}GB')
    except Exception: pass

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for g in gpus:
        try: tf.config.experimental.set_memory_growth(g, True)
        except Exception as e: print(f'Memory growth failed: {e}')
    
    # CRITICAL P100 FIX: Disable mixed precision as P100 lacks hardware Tensor Cores
    tf.keras.mixed_precision.set_global_policy('float32') 
    print('✅ Standard float32 precision enforced for P100 architecture')
    
    # Enable XLA Compiler for speed boost
    try: 
        tf.config.optimizer.set_jit(True)
        print('✅ XLA compiler enabled')
    except Exception as e: print(f'XLA failed: {e}')
else:
    print('❌ GPU not found. Enable Kaggle GPU accelerator in the top right menu.')

print_ram('Initial RAM')

# ============================================================
# DATASET DISCOVERY & FOLDER SETUP
# ============================================================

print("\nDownloading Dataset...")
dataset_root = kagglehub.dataset_download("nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20")
print(f"✅ Downloaded to: {dataset_root}")

def find_subdir(root_path, target_name):
    try:
        return str(next(Path(root_path).rglob(target_name)))
    except StopIteration:
        return None

TRAIN_DIR = find_subdir(dataset_root, "train")
VAL_DIR   = find_subdir(dataset_root, "val")
TEST_DIR  = find_subdir(dataset_root, "test")

assert TRAIN_DIR and VAL_DIR and TEST_DIR, "Missing train/val/test folders"
print(f"Train: {TRAIN_DIR}\nVal:   {VAL_DIR}\nTest:  {TEST_DIR}")

OUTPUT_DIR = "/kaggle/working/brain_tumor_results_advanced_bigv10.1 all"
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
PRED_DIR = os.path.join(OUTPUT_DIR, 'predictions')
STAT_DIR = os.path.join(OUTPUT_DIR, 'statistics')
GRADCAM_DIR = os.path.join(OUTPUT_DIR, 'gradcam')
CKPT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
LOG_DIR = os.path.join(OUTPUT_DIR, 'logs')
META_DIR = os.path.join(OUTPUT_DIR, 'meta_learner')
for d in [MODEL_DIR, FIG_DIR, PRED_DIR, STAT_DIR, GRADCAM_DIR, CKPT_DIR, LOG_DIR, META_DIR]: os.makedirs(d, exist_ok=True)

def get_class_names(train_dir):
    fp = os.path.join(os.path.dirname(train_dir), 'class_names.txt')
    if os.path.exists(fp):
        return [x.strip() for x in open(fp, encoding='utf-8') if x.strip()]
    return sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir,d))])

def list_paths_and_labels(split_dir, class_names):
    paths, labels = [], []
    for i,c in enumerate(class_names):
        cd = os.path.join(split_dir,c)
        if not os.path.isdir(cd): continue
        for p in Path(cd).rglob('*'):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                paths.append(str(p)); labels.append(i)
    return np.array(paths), np.array(labels, dtype=np.int32)

CLASS_NAMES = get_class_names(TRAIN_DIR)
NUM_CLASSES = len(CLASS_NAMES)
for i,c in enumerate(CLASS_NAMES):
    if 'meningioma' in c.lower(): MENINGIOMA_CLASS_IDX = i
train_paths, train_labels = list_paths_and_labels(TRAIN_DIR, CLASS_NAMES)
val_paths, val_labels = list_paths_and_labels(VAL_DIR, CLASS_NAMES)
test_paths, test_labels = list_paths_and_labels(TEST_DIR, CLASS_NAMES)
print('Classes:', CLASS_NAMES)
print('Meningioma index:', MENINGIOMA_CLASS_IDX, CLASS_NAMES[MENINGIOMA_CLASS_IDX])
print(f'Train={len(train_paths)} | Val={len(val_paths)} | Test={len(test_paths)}')
pd.DataFrame({'split':['train','val','test'],'n_images':[len(train_paths),len(val_paths),len(test_paths)]}).to_csv(os.path.join(STAT_DIR,'dataset_split_counts.csv'), index=False)

# ============================================================
# MODEL BUILDING
# ============================================================

def identity_preprocess(x): return x
PREPROCESS_DICT = {'mobilenetv2':preprocess_mnv2,'mobilenetv3':identity_preprocess,'resnet50v2':preprocess_resnetv2,'efficientnetb0':preprocess_eff,'efficientnetb4':preprocess_eff,'vgg16':preprocess_vgg16}

def get_batch_size(m): return BATCH_SIZE_DICT.get(m, 16)

def get_backbone(model_name, inp, input_shape):
    try:
        if model_name=='mobilenetv2': return MobileNetV2(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
        if model_name=='mobilenetv3': return MobileNetV3Large(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape, include_preprocessing=True)
        if model_name=='resnet50v2': return ResNet50V2(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
        if model_name=='efficientnetb0': return EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
        if model_name=='efficientnetb4': return EfficientNetB4(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
        if model_name=='vgg16': return VGG16(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
    except Exception as e:
        print(f'ImageNet weights unavailable for {model_name}: {e}\nFalling back to weights=None')
    if model_name=='mobilenetv2': return MobileNetV2(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    if model_name=='mobilenetv3': return MobileNetV3Large(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape, include_preprocessing=True)
    if model_name=='resnet50v2': return ResNet50V2(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    if model_name=='efficientnetb0': return EfficientNetB0(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    if model_name=='efficientnetb4': return EfficientNetB4(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    if model_name=='vgg16': return VGG16(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    raise ValueError(model_name)

def build_model(model_name):
    input_shape = IMG_SIZE_DICT[model_name] + (3,)
    inp = keras.Input(shape=input_shape)
    base = get_backbone(model_name, inp, input_shape)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(DENSE_UNITS, activation='relu', kernel_regularizer=keras.regularizers.l2(WEIGHT_DECAY))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    return keras.Model(inp, out, name=model_name), base

def make_optimizer(lr): return keras.optimizers.Adam(learning_rate=lr, global_clipnorm=GRADIENT_CLIP_NORM)

class SparseLabelSmoothingLoss(tf.keras.losses.Loss):
    def __init__(self, num_classes, smoothing=0.1, **kw): super().__init__(**kw); self.num_classes=num_classes; self.smoothing=smoothing
    def call(self, y_true, y_pred):
        y = tf.one_hot(tf.cast(y_true, tf.int32), self.num_classes)
        y = y*(1-self.smoothing) + self.smoothing/self.num_classes
        return tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y, y_pred))

def get_loss_fn():
    return SparseLabelSmoothingLoss(NUM_CLASSES, LABEL_SMOOTHING_VALUE) if USE_LABEL_SMOOTHING else 'sparse_categorical_crossentropy'

# ============================================================
# TF.DATA PIPELINE
# ============================================================

def oversample_meningioma(paths, labels, seed=SEED):
    idx = np.where(labels == MENINGIOMA_CLASS_IDX)[0]
    n_extra = int(len(idx) * MENINGIOMA_OVERSAMPLE_FACTOR)
    if n_extra <= 0 or len(idx)==0: return paths, labels
    rng = np.random.default_rng(seed)
    extra = rng.choice(idx, size=n_extra, replace=True)
    print(f'Meningioma oversampling: added {n_extra}')
    return np.concatenate([paths, paths[extra]]), np.concatenate([labels, labels[extra]])

def decode_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None,None,3])
    return tf.cast(img, tf.float32)

def augment_image(img, y):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.08)
    img = tf.image.random_contrast(img, 0.92, 1.08)
    return tf.clip_by_value(img, 0, 255), y

def make_dataset(paths, labels, model_name, shuffle=False, augment=False, batch_size=None):
    if batch_size is None: batch_size = get_batch_size(model_name)
    p, y = np.array(paths), np.array(labels, dtype=np.int32)
    if augment and MENINGIOMA_OVERSAMPLE_FACTOR > 0: p, y = oversample_meningioma(p, y)
    size = IMG_SIZE_DICT[model_name]; pre = PREPROCESS_DICT[model_name]
    ds = tf.data.Dataset.from_tensor_slices((p,y))
    if shuffle: ds = ds.shuffle(min(len(p),1000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda fp, lab: (tf.image.resize_with_pad(decode_image(fp), size[0], size[1]), lab), num_parallel_calls=AUTOTUNE)
    if augment and USE_ONLINE_AUGMENTATION: ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.map(lambda x, lab: (pre(x), lab), num_parallel_calls=AUTOTUNE)
    opt = tf.data.Options(); opt.experimental_deterministic = not shuffle
    return ds.with_options(opt).prefetch(AUTOTUNE)

# ============================================================
# METRICS AND PLOTS (INCLUDES IOU/JACCARD)
# ============================================================

def predict_dataset(model, ds):
    yt, yp, pr = [], [], []
    for x,y in ds:
        p = model.predict(x, verbose=0)
        yt.append(y.numpy()); yp.append(np.argmax(p, axis=1)); pr.append(p)
    return np.concatenate(yt), np.concatenate(yp), np.vstack(pr)

def safe_auc(y_true, probs):
    try: return skm.roc_auc_score(y_true, probs, multi_class='ovr', average='macro')
    except Exception: return np.nan

def compute_metrics(y_true, y_pred, probs=None, prefix='test'):
    r = {
        f'{prefix}_accuracy': skm.accuracy_score(y_true, y_pred), 
        f'{prefix}_macro_f1': skm.f1_score(y_true, y_pred, average='macro', zero_division=0), 
        f'{prefix}_macro_precision': skm.precision_score(y_true, y_pred, average='macro', zero_division=0), 
        f'{prefix}_macro_recall': skm.recall_score(y_true, y_pred, average='macro', zero_division=0), 
        f'{prefix}_weighted_f1': skm.f1_score(y_true, y_pred, average='weighted', zero_division=0),
        f'{prefix}_iou_jaccard': skm.jaccard_score(y_true, y_pred, average='macro') # IoU Implementation
    }
    if probs is not None: r[f'{prefix}_auc'] = safe_auc(y_true, probs)
    return r

def per_class_metrics(y_true, y_pred, model_name):
    cm = skm.confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    rows=[]
    for i,c in enumerate(CLASS_NAMES):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        prec=tp/(tp+fp) if tp+fp else 0; rec=tp/(tp+fn) if tp+fn else 0; spec=tn/(tn+fp) if tn+fp else 0; f1=2*prec*rec/(prec+rec) if prec+rec else 0
        rows.append({'model':model_name,'class':c,'precision':prec,'recall_sensitivity':rec,'specificity':spec,'f1_score':f1,'support':int(tp+fn)})
    return pd.DataFrame(rows)

def bootstrap_ci(y_true, y_pred, func, n_boot=1000):
    rng=np.random.default_rng(SEED); n=len(y_true); vals=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n); vals.append(func(y_true[idx], y_pred[idx]))
    lo,hi=np.percentile(vals,[2.5,97.5]); return float(lo), float(hi)

def compute_bootstrap_table(y_true, y_pred, model_name):
    funcs={
        'accuracy': skm.accuracy_score,
        'macro_f1': lambda a,b: skm.f1_score(a,b,average='macro',zero_division=0),
        'macro_precision': lambda a,b: skm.precision_score(a,b,average='macro',zero_division=0),
        'macro_recall': lambda a,b: skm.recall_score(a,b,average='macro',zero_division=0),
        'iou_jaccard': lambda a,b: skm.jaccard_score(a,b,average='macro') # IoU Implementation
    }
    rows=[]
    for k,f in funcs.items():
        est=f(y_true,y_pred); lo,hi=bootstrap_ci(y_true,y_pred,f)
        rows.append({'model':model_name,'metric':k,'estimate':est,'ci_low':lo,'ci_high':hi})
    return pd.DataFrame(rows)

def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(8,7)); plt.imshow(cm); plt.title(title); plt.xlabel('Predicted'); plt.ylabel('True')
    plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha='right'); plt.yticks(range(NUM_CLASSES), CLASS_NAMES)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): plt.text(j,i,str(cm[i,j]),ha='center',va='center')
    plt.colorbar(); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, filename), dpi=300); plt.close()

def plot_multiclass_roc(y_true, probs, title, filename):
    ybin=label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    plt.figure(figsize=(8,7))
    for i,c in enumerate(CLASS_NAMES):
        try:
            fpr,tpr,_=skm.roc_curve(ybin[:,i], probs[:,i]); auc=skm.auc(fpr,tpr); plt.plot(fpr,tpr,label=f'{c} AUC={auc:.3f}')
        except Exception: pass
    plt.plot([0,1],[0,1],linestyle='--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title(title); plt.legend(loc='lower right',fontsize=8); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,filename), dpi=300); plt.close()

def plot_training_curves(model_name):
    dfs=[]; offset=0
    for phase in ['phase1','phase2']:
        fp=os.path.join(LOG_DIR,f'{model_name}_{phase}_log.csv')
        if os.path.exists(fp):
            df=pd.read_csv(fp); df['epoch_global']=np.arange(len(df))+1+offset; offset+=len(df); dfs.append(df)
    if not dfs: return
    h=pd.concat(dfs, ignore_index=True)
    if 'accuracy' in h and 'val_accuracy' in h:
        plt.figure(figsize=(8,6)); plt.plot(h.epoch_global,h.accuracy,label='Train Accuracy'); plt.plot(h.epoch_global,h.val_accuracy,label='Validation Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title(f'{model_name} Accuracy Curve'); plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,f'{model_name}_accuracy_curve.png'),dpi=300); plt.close()
    if 'loss' in h and 'val_loss' in h:
        plt.figure(figsize=(8,6)); plt.plot(h.epoch_global,h.loss,label='Train Loss'); plt.plot(h.epoch_global,h.val_loss,label='Validation Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'{model_name} Loss Curve'); plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,f'{model_name}_loss_curve.png'),dpi=300); plt.close()

def save_prediction_csv(split, model_name, y_true, y_pred, probs, paths):
    d={'image_path':paths,'true_label_index':y_true,'pred_label_index':y_pred,'true_label':[CLASS_NAMES[i] for i in y_true],'pred_label':[CLASS_NAMES[i] for i in y_pred]}
    for i,c in enumerate(CLASS_NAMES): d[f'prob_{c}']=probs[:,i]
    pd.DataFrame(d).to_csv(os.path.join(PRED_DIR,f'{model_name}_{split}_predictions.csv'), index=False)

# ============================================================
# FOCUSED GRAD-CAM++
# ============================================================

def find_last_conv_layer(model):
    convs=(layers.Conv2D,layers.DepthwiseConv2D,layers.SeparableConv2D)
    for l in reversed(model.layers):
        if isinstance(l, convs): return l.name
    for l in reversed(model.layers):
        try:
            if len(l.output.shape)==4: return l.name
        except Exception: pass
    return None

def normalize_heatmap(h):
    h=np.nan_to_num(h); h=np.maximum(h,0); mx=np.max(h)
    return (h/mx).astype(np.float32) if mx>0 else h.astype(np.float32)

def make_gradcam_heatmap(model, img_batch, last_conv, pred_idx):
    gm=keras.Model(model.inputs, [model.get_layer(last_conv).output, model.output])
    with tf.GradientTape() as tape:
        conv,pred=gm(img_batch, training=False); loss=pred[:,pred_idx]
    grads=tape.gradient(loss, conv); weights=tf.reduce_mean(grads, axis=(0,1,2)); cam=tf.reduce_sum(conv[0]*weights, axis=-1)
    return normalize_heatmap(tf.nn.relu(cam).numpy())

def make_gradcampp_heatmap(model, img_batch, last_conv, pred_idx):
    gm=keras.Model(model.inputs, [model.get_layer(last_conv).output, model.output])
    with tf.GradientTape() as tape:
        conv,pred=gm(img_batch, training=False); loss=pred[:,pred_idx]
    grads=tape.gradient(loss, conv)[0].numpy(); conv=conv[0].numpy()
    first=grads; second=grads**2; third=grads**3; gsum=np.sum(conv, axis=(0,1), keepdims=True)
    denom=2*second + third*gsum; denom=np.where(denom!=0, denom, 1e-8)
    alphas=second/denom; weights=np.sum(alphas*np.maximum(first,0), axis=(0,1))
    return normalize_heatmap(np.sum(weights*conv, axis=-1))

def focus_heatmap(h, percentile=85):
    h=normalize_heatmap(h)
    if np.max(h)<=0: return h
    thr=np.percentile(h, percentile); mask=(h>=thr).astype(np.uint8)
    try:
        n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        if n>1:
            largest=1+int(np.argmax(stats[1:, cv2.CC_STAT_AREA])); mask=(lab==largest).astype(np.uint8)
    except Exception: pass
    out=h*mask
    if out.shape[0]>=5 and out.shape[1]>=5: out=cv2.GaussianBlur(out,(5,5),0)
    return normalize_heatmap(out)

def overlay_heatmap(img, h, alpha=0.30):
    img=img.astype(np.uint8); hh,ww=img.shape[:2]
    h=cv2.resize(normalize_heatmap(h),(ww,hh)); h=np.uint8(255*h)
    col=cv2.applyColorMap(h, cv2.COLORMAP_JET); col=cv2.cvtColor(col, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img,1-alpha,col,alpha,0)

def load_image_for_cam(path, model_name):
    size=IMG_SIZE_DICT[model_name]; pre=PREPROCESS_DICT[model_name]
    raw=tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False); raw.set_shape([None,None,3]); raw=tf.cast(raw, tf.float32); raw=tf.image.resize_with_pad(raw,size[0],size[1])
    raw_np=tf.cast(raw, tf.uint8).numpy(); batch=pre(tf.expand_dims(raw,0))
    return raw_np,batch

def select_balanced_indices(labels, n=8):
    rng=np.random.default_rng(SEED); idx=[]; per=max(1, math.ceil(n/NUM_CLASSES)); labels=np.array(labels)
    for c in range(NUM_CLASSES):
        ci=np.where(labels==c)[0]
        if len(ci): idx += rng.choice(ci, size=min(per,len(ci)), replace=False).tolist()
    return rng.choice(idx, size=min(n,len(idx)), replace=False).tolist() if len(idx)>n else idx

def plot_gradcam_individual_model(model, model_name, paths, labels_array, tag='test', n=8):
    last=find_last_conv_layer(model)
    if not last: print('No conv layer for', model_name); return
    idxs=select_balanced_indices(labels_array,n); cols=4; rows=math.ceil(len(idxs)/cols)
    fig,axes=plt.subplots(rows,cols,figsize=(16,4*rows)); axes=np.array(axes).reshape(-1)
    for ax_i,idx in enumerate(idxs):
        raw,b=load_image_for_cam(paths[idx], model_name); probs=model.predict(b,verbose=0)[0]; pred=int(np.argmax(probs))
        h=make_gradcampp_heatmap(model,b,last,pred) if USE_GRADCAM_PLUS_PLUS else make_gradcam_heatmap(model,b,last,pred)
        h=focus_heatmap(h, GRADCAM_THRESHOLD_PERCENTILE); over=overlay_heatmap(raw,h,GRADCAM_ALPHA)
        true=int(labels_array[idx]); axes[ax_i].imshow(over); axes[ax_i].set_title(f'True: {CLASS_NAMES[true]}\nPred: {CLASS_NAMES[pred]} ({probs[pred]:.2f})', color='green' if pred==true else 'red'); axes[ax_i].axis('off')
    for j in range(len(idxs), len(axes)): axes[j].axis('off')
    plt.tight_layout(); plt.savefig(os.path.join(GRADCAM_DIR,f'{model_name}_{tag}_focused_gradcampp.png'),dpi=300); plt.close()

def generate_ensemble_gradcam(ensemble_name, model_names, model_weights, target_preds, paths, labels_array, average_mode=False, n=8):
    idxs=select_balanced_indices(labels_array,n); size=IMG_SIZE_DICT[model_names[0]]
    fused={i:np.zeros(size,dtype=np.float32) for i in idxs}; raw_imgs={}
    weights=np.ones(len(model_names))/len(model_names) if average_mode else np.array(model_weights,dtype=np.float32)/(np.sum(model_weights)+1e-8)
    for m,w in zip(model_names,weights):
        wf=os.path.join(MODEL_DIR,f'{m}_final_best.weights.h5')
        if not os.path.exists(wf): continue
        model,_=build_model(m); model.load_weights(wf); last=find_last_conv_layer(model)
        if not last: continue
        for idx in idxs:
            raw,b=load_image_for_cam(paths[idx],m); raw_imgs.setdefault(idx, raw); target=int(target_preds[idx])
            h=make_gradcampp_heatmap(model,b,last,target) if USE_GRADCAM_PLUS_PLUS else make_gradcam_heatmap(model,b,last,target)
            fused[idx]+=float(w)*cv2.resize(normalize_heatmap(h),(size[1],size[0]))
        del model; tf.keras.backend.clear_session(); gc.collect()
    cols=4; rows=math.ceil(len(idxs)/cols); fig,axes=plt.subplots(rows,cols,figsize=(16,4*rows)); axes=np.array(axes).reshape(-1)
    for ax_i,idx in enumerate(idxs):
        true=int(labels_array[idx]); pred=int(target_preds[idx]); h=focus_heatmap(fused[idx], GRADCAM_THRESHOLD_PERCENTILE); over=overlay_heatmap(raw_imgs[idx],h,GRADCAM_ALPHA)
        axes[ax_i].imshow(over); axes[ax_i].set_title(f'True: {CLASS_NAMES[true]}\nEnsemble Pred: {CLASS_NAMES[pred]}', color='green' if pred==true else 'red'); axes[ax_i].axis('off')
    for j in range(len(idxs),len(axes)): axes[j].axis('off')
    plt.tight_layout(); plt.savefig(os.path.join(GRADCAM_DIR,f'{ensemble_name}_focused_ensemble_gradcampp.png'),dpi=300); plt.close()

# ============================================================
# CROSS-VALIDATION
# ============================================================

def run_kfold(model_name):
    print(f'\n========== {N_SPLITS}-FOLD CV: {model_name} ==========')
    summary_fp=os.path.join(STAT_DIR,f'{model_name}_cv_summary.json'); fold_fp=os.path.join(STAT_DIR,f'{model_name}_cv_fold_results.csv')
    if os.path.exists(summary_fp) and os.path.exists(fold_fp) and not FORCE_RETRAIN:
        return json.load(open(summary_fp))
    rows=[]; done=set()
    if os.path.exists(fold_fp) and not FORCE_RETRAIN:
        old=pd.read_csv(fold_fp); rows=old.to_dict('records'); done=set(old.fold.astype(int))
    skf=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fold,(tr,va) in enumerate(skf.split(train_paths, train_labels),1):
        if fold in done: print('Skip completed fold',fold); continue
        model=None
        try:
            print(f'Fold {fold}/{N_SPLITS} - {model_name}')
            tr_ds=make_dataset(train_paths[tr], train_labels[tr], model_name, shuffle=True, augment=True)
            tr_eval=make_dataset(train_paths[tr], train_labels[tr], model_name, shuffle=False, augment=False)
            va_ds=make_dataset(train_paths[va], train_labels[va], model_name, shuffle=False, augment=False)
            model,base=build_model(model_name); base.trainable=False
            model.compile(optimizer=make_optimizer(LR_HEAD), loss=get_loss_fn(), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
            wfp=os.path.join(MODEL_DIR,f'{model_name}_fold_{fold}_best.weights.h5')
            callbacks=[keras.callbacks.ModelCheckpoint(wfp,monitor='val_accuracy',mode='max',save_best_only=True,save_weights_only=True,verbose=1), keras.callbacks.BackupAndRestore(os.path.join(CKPT_DIR,f'{model_name}_fold_{fold}_backup'), save_freq='epoch'), keras.callbacks.CSVLogger(os.path.join(LOG_DIR,f'{model_name}_fold_{fold}_log.csv'), append=True), keras.callbacks.EarlyStopping(monitor='val_accuracy',mode='max',patience=EARLY_STOPPING_PATIENCE,restore_best_weights=True,verbose=1)]
            model.fit(tr_ds, validation_data=va_ds, epochs=HEAD_EPOCHS, verbose=1, callbacks=callbacks)
            if os.path.exists(wfp): model.load_weights(wfp)
            yt,yp,pr=predict_dataset(model,tr_eval); vt,vp,vpr=predict_dataset(model,va_ds)
            tm=compute_metrics(yt,yp,pr,'train'); vm=compute_metrics(vt,vp,vpr,'val')
            plot_multiclass_roc(vt,vpr,f'{model_name} Fold-{fold} ROC',f'{model_name}_fold_{fold}_roc.png')
            rows.append({'model':model_name,'fold':fold,**tm,**vm}); pd.DataFrame(rows).to_csv(fold_fp,index=False)
            print(f"Fold {fold}: Train Acc={tm['train_accuracy']:.4f}, Train F1={tm['train_macro_f1']:.4f}, Train IoU={tm['train_iou_jaccard']:.4f}, Val Acc={vm['val_accuracy']:.4f}, Val F1={vm['val_macro_f1']:.4f}, Val IoU={vm['val_iou_jaccard']:.4f}")
        except Exception as e:
            print(f'CV error {model_name} fold {fold}:', e)
        finally:
            if model is not None: del model
            tf.keras.backend.clear_session(); gc.collect(); print_ram(f'After CV {model_name} fold {fold}')
    df=pd.DataFrame(rows)
    if df.empty: return {'model':model_name}
    summary={'model':model_name,'train_accuracy_mean':float(df.train_accuracy.mean()),'train_accuracy_std':float(df.train_accuracy.std()),'train_macro_f1_mean':float(df.train_macro_f1.mean()),'train_macro_f1_std':float(df.train_macro_f1.std()),'train_iou_jaccard_mean':float(df.train_iou_jaccard.mean()),'val_accuracy_mean':float(df.val_accuracy.mean()),'val_accuracy_std':float(df.val_accuracy.std()),'val_macro_f1_mean':float(df.val_macro_f1.mean()),'val_macro_f1_std':float(df.val_macro_f1.std()),'val_iou_jaccard_mean':float(df.val_iou_jaccard.mean()),'val_auc_mean':float(df.val_auc.mean()),'val_auc_std':float(df.val_auc.std()),'completed_folds':int(df.fold.nunique())}
    json.dump(summary, open(summary_fp,'w'), indent=2); return summary

# ============================================================
# FINAL TRAINING
# ============================================================

def final_summary_path(m): return os.path.join(STAT_DIR,f'{m}_final_summary.json')
def final_weight_path(m): return os.path.join(MODEL_DIR,f'{m}_final_best.weights.h5')
def final_probs_exist(m): return all(os.path.exists(os.path.join(PRED_DIR,f'{m}_{s}_{t}.npy')) for s in ['train','val','test'] for t in ['probs','preds'])

def evaluate_and_save_final_model(model, m):
    splits=[('train',train_paths,train_labels),('val',val_paths,val_labels),('test',test_paths,test_labels)]
    allm={}
    for split,paths,labels in splits:
        ds=make_dataset(paths,labels,m,shuffle=False,augment=False)
        yt,yp,pr=predict_dataset(model,ds)
        np.save(os.path.join(PRED_DIR,f'{m}_{split}_probs.npy'),pr); np.save(os.path.join(PRED_DIR,f'{m}_{split}_preds.npy'),yp)
        save_prediction_csv(split,m,yt,yp,pr,paths)
        allm.update(compute_metrics(yt,yp,pr,split))
        if split in ['val','test']: plot_multiclass_roc(yt,pr,f'{m} {split} ROC',f'{m}_{split}_roc.png')
        if split=='test':
            cm=skm.confusion_matrix(yt,yp,labels=list(range(NUM_CLASSES))); plot_confusion_matrix(cm,f'{m} Test Confusion Matrix',f'{m}_test_cm.png')
            per_class_metrics(yt,yp,m).to_csv(os.path.join(STAT_DIR,f'{m}_per_class_metrics.csv'),index=False)
            compute_bootstrap_table(yt,yp,m).to_csv(os.path.join(STAT_DIR,f'{m}_bootstrap_ci.csv'),index=False)
    summary={'model':m,**allm}; json.dump(summary,open(final_summary_path(m),'w'),indent=2)
    print(f"{m}: Train Acc={allm['train_accuracy']:.4f}, Val Acc={allm['val_accuracy']:.4f}, Test Acc={allm['test_accuracy']:.4f}, Test F1={allm['test_macro_f1']:.4f}, Test IoU={allm['test_iou_jaccard']:.4f}")
    return summary

def train_final_model(m):
    print(f'\n===== FINAL TRAINING: {m} =====')
    print('Label Smoothing + Meningioma Oversampling | Class Weighting OFF | Focal Loss OFF')
    if os.path.exists(final_summary_path(m)) and os.path.exists(final_weight_path(m)) and final_probs_exist(m) and not FORCE_RETRAIN and not FORCE_REEVALUATE:
        print('Already completed:', m); return json.load(open(final_summary_path(m)))
    model=None
    try:
        tr_ds=make_dataset(train_paths,train_labels,m,shuffle=True,augment=True); va_ds=make_dataset(val_paths,val_labels,m,shuffle=False,augment=False)
        model,base=build_model(m)
        if os.path.exists(final_weight_path(m)) and not FORCE_RETRAIN:
            model.load_weights(final_weight_path(m)); summary=evaluate_and_save_final_model(model,m)
            if RUN_GRADCAM: plot_gradcam_individual_model(model,m,test_paths,test_labels,'test',GRADCAM_N_IMAGES)
            return summary
        base.trainable=False; model.compile(optimizer=make_optimizer(LR_HEAD), loss=get_loss_fn(), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
        p1=os.path.join(MODEL_DIR,f'{m}_phase1_best.weights.h5')
        cb1=[keras.callbacks.ModelCheckpoint(p1,monitor='val_accuracy',mode='max',save_best_only=True,save_weights_only=True,verbose=1),keras.callbacks.BackupAndRestore(os.path.join(CKPT_DIR,f'{m}_phase1_backup'),save_freq='epoch'),keras.callbacks.CSVLogger(os.path.join(LOG_DIR,f'{m}_phase1_log.csv'),append=True),keras.callbacks.EarlyStopping(monitor='val_accuracy',mode='max',patience=EARLY_STOPPING_PATIENCE,restore_best_weights=True,verbose=1)]
        model.fit(tr_ds,validation_data=va_ds,epochs=HEAD_EPOCHS,verbose=1,callbacks=cb1)
        if os.path.exists(p1): model.load_weights(p1)
        model.save_weights(os.path.join(MODEL_DIR,f'{m}_after_phase1.weights.h5'))
        base.trainable=True
        for layer in base.layers[:-50]: layer.trainable=False
        model.compile(optimizer=make_optimizer(LR_FINE), loss=get_loss_fn(), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
        cb2=[keras.callbacks.ModelCheckpoint(final_weight_path(m),monitor='val_accuracy',mode='max',save_best_only=True,save_weights_only=True,verbose=1),keras.callbacks.BackupAndRestore(os.path.join(CKPT_DIR,f'{m}_phase2_backup'),save_freq='epoch'),keras.callbacks.CSVLogger(os.path.join(LOG_DIR,f'{m}_phase2_log.csv'),append=True),keras.callbacks.EarlyStopping(monitor='val_accuracy',mode='max',patience=EARLY_STOPPING_PATIENCE,restore_best_weights=True,verbose=1)]
        model.fit(tr_ds,validation_data=va_ds,epochs=FINE_TUNE_EPOCHS,verbose=1,callbacks=cb2)
        if os.path.exists(final_weight_path(m)): model.load_weights(final_weight_path(m))
        else: model.save_weights(final_weight_path(m))
        summary=evaluate_and_save_final_model(model,m); plot_training_curves(m)
        if RUN_GRADCAM: plot_gradcam_individual_model(model,m,test_paths,test_labels,'test',GRADCAM_N_IMAGES)
        return summary
    except Exception as e:
        print('Training error:',m,e); return {'model':m,'error':str(e)}
    finally:
        if model is not None: del model
        tf.keras.backend.clear_session(); gc.collect(); print_ram(f'After final {m}')

# ============================================================
# ENSEMBLES
# ============================================================

def load_probs(models, split): return [np.load(os.path.join(PRED_DIR,f'{m}_{split}_probs.npy')) for m in models]
def split_labels(split): return {'train':train_labels,'val':val_labels,'test':test_labels}[split]
def split_paths(split): return {'train':train_paths,'val':val_paths,'test':test_paths}[split]

def save_ensemble(split,name,probs,preds):
    np.save(os.path.join(PRED_DIR,f'{name}_{split}_probs.npy'),probs); np.save(os.path.join(PRED_DIR,f'{name}_{split}_preds.npy'),preds)
    save_prediction_csv(split,name,split_labels(split),preds,probs,split_paths(split))

def evaluate_method(name, trp, vap, tep):
    rows={'model':name}
    for split,probs in [('train',trp),('val',vap),('test',tep)]:
        preds=np.argmax(probs,axis=1); save_ensemble(split,name,probs,preds); rows.update(compute_metrics(split_labels(split),preds,probs,split))
        if split in ['val','test']: plot_multiclass_roc(split_labels(split),probs,f'{name} {split} ROC',f'{name}_{split}_roc.png')
        if split=='test':
            cm=skm.confusion_matrix(test_labels,preds,labels=list(range(NUM_CLASSES))); plot_confusion_matrix(cm,f'{name} Test Confusion Matrix',f'{name}_test_cm.png')
            per_class_metrics(test_labels,preds,name).to_csv(os.path.join(STAT_DIR,f'{name}_per_class_metrics.csv'),index=False)
            compute_bootstrap_table(test_labels,preds,name).to_csv(os.path.join(STAT_DIR,f'{name}_bootstrap_ci.csv'),index=False)
    print(f"{name}: Train Acc={rows['train_accuracy']:.4f}, Val Acc={rows['val_accuracy']:.4f}, Test Acc={rows['test_accuracy']:.4f}, Test F1={rows['test_macro_f1']:.4f}, Test IoU={rows['test_iou_jaccard']:.4f}")
    return rows, np.argmax(tep,axis=1)

def run_weighted_soft_voting(models, metrics_df, name='weighted_soft_voting_ensemble'):
    weights=np.array([float(metrics_df[metrics_df.model==m].iloc[0].val_macro_f1) if len(metrics_df[metrics_df.model==m]) else 1.0 for m in models], dtype=np.float32); weights=weights/(weights.sum()+1e-8)
    pd.DataFrame({'model':models,'weight_from_validation_macro_f1':weights}).to_csv(os.path.join(STAT_DIR,f'{name}_weights.csv'),index=False)
    outs=[]
    for split in ['train','val','test']:
        plist=load_probs(models,split); out=np.zeros_like(plist[0],dtype=np.float32)
        for w,p in zip(weights,plist): out += float(w)*p
        outs.append(out)
    row,pred=evaluate_method(name,*outs); row['selected_models']=', '.join(models); row['weighting_strategy']='validation_macro_f1'
    return row, weights, pred

def stacking_features(models, split): return np.hstack(load_probs(models,split))
def align_meta_proba(probs, classes):
    out=np.zeros((probs.shape[0],NUM_CLASSES),dtype=np.float32)
    for i,c in enumerate(classes): out[:,int(c)]=probs[:,i]
    return out

def run_stacking_ensemble(models):
    Xtr=stacking_features(models,'train'); Xva=stacking_features(models,'val'); Xte=stacking_features(models,'test')
    meta=LogisticRegression(max_iter=5000,C=1.0,solver='lbfgs',multi_class='auto',random_state=SEED)
    meta.fit(Xva, val_labels)  # test is not used for fitting
    joblib.dump(meta, os.path.join(META_DIR,'stacking_logistic_regression_meta_learner.joblib'))
    trp=align_meta_proba(meta.predict_proba(Xtr), meta.classes_); vap=align_meta_proba(meta.predict_proba(Xva), meta.classes_); tep=align_meta_proba(meta.predict_proba(Xte), meta.classes_)
    row,pred=evaluate_method('stacking_logistic_regression_ensemble',trp,vap,tep); row['selected_models']=', '.join(models); row['meta_learner']='LogisticRegression'; row['meta_training_data']='validation_probability_vectors'
    return row,pred

def run_selective_topk(metrics_df):
    rows=[]; ranked=metrics_df.sort_values('val_macro_f1',ascending=False)
    for k in SELECTIVE_TOP_K_LIST:
        if k>len(ranked): continue
        models=ranked.head(k).model.tolist(); row,_,_=run_weighted_soft_voting(models, ranked, f'selective_weighted_soft_voting_top_{k}'); rows.append(row)
    return rows

# ============================================================
# MCNEMAR
# ============================================================

def mcnemar_pairwise(pred_dict, y):
    rows=[]; names=list(pred_dict)
    for i,a in enumerate(names):
        for b in names[i+1:]:
            pa,pb=pred_dict[a],pred_dict[b]; ac=pa==y; bc=pb==y
            both=int(np.sum(ac&bc)); aw=int(np.sum(ac&~bc)); bw=int(np.sum(~ac&bc)); none=int(np.sum(~ac&~bc))
            try:
                res=mcnemar([[both,aw],[bw,none]], exact=False, correction=True); stat=float(res.statistic); p=float(res.pvalue)
            except Exception: stat=np.nan; p=np.nan
            rows.append({'model_a':a,'model_b':b,'both_correct':both,'a_correct_b_wrong':aw,'a_wrong_b_correct':bw,'both_wrong':none,'mcnemar_statistic':stat,'p_value':p,'significant_at_0_05':bool(p<0.05) if not np.isnan(p) else False})
    df=pd.DataFrame(rows)
    if len(df):
        df['bonferroni_alpha']=0.05/len(df); df['significant_after_bonferroni']=df.p_value<df.bonferroni_alpha
    return df

# ============================================================
# MAIN
# ============================================================

def main():
    print('\nPIPELINE STARTED:', datetime.now())
    cv_rows=[]; final_rows=[]
    if RUN_KFOLD:
        for m in MODELS_TO_RUN:
            try: cv_rows.append(run_kfold(m))
            except Exception as e: print('CV failed:',m,e)
            tf.keras.backend.clear_session(); gc.collect()
        if cv_rows: pd.DataFrame(cv_rows).to_csv(os.path.join(STAT_DIR,'cv_summary_results.csv'),index=False)
    
    if RUN_FINAL_TRAINING:
        for m in MODELS_TO_RUN:
            final_rows.append(train_final_model(m))
            pd.DataFrame(final_rows).to_csv(os.path.join(STAT_DIR,'individual_model_train_val_test_results_partial.csv'),index=False)
            tf.keras.backend.clear_session(); gc.collect()
        
        metrics_df=pd.DataFrame([r for r in final_rows if isinstance(r,dict) and 'error' not in r and 'test_accuracy' in r])
        metrics_df.to_csv(os.path.join(STAT_DIR,'individual_model_train_val_test_results.csv'),index=False)
        
        pcs=[os.path.join(STAT_DIR,f'{m}_per_class_metrics.csv') for m in MODELS_TO_RUN if os.path.exists(os.path.join(STAT_DIR,f'{m}_per_class_metrics.csv'))]
        if pcs: pd.concat([pd.read_csv(f) for f in pcs],ignore_index=True).to_csv(os.path.join(STAT_DIR,'all_individual_per_class_metrics.csv'),index=False)
        
        cis=[os.path.join(STAT_DIR,f'{m}_bootstrap_ci.csv') for m in MODELS_TO_RUN if os.path.exists(os.path.join(STAT_DIR,f'{m}_bootstrap_ci.csv'))]
        if cis: pd.concat([pd.read_csv(f) for f in cis],ignore_index=True).to_csv(os.path.join(STAT_DIR,'all_individual_bootstrap_cis.csv'),index=False)
    else:
        metrics_df=pd.read_csv(os.path.join(STAT_DIR,'individual_model_train_val_test_results.csv'))
    
    available=[m for m in MODELS_TO_RUN if final_probs_exist(m)]
    if len(available)<2: 
        print('Not enough models for ensemble'); return
    
    ensemble_rows=[]; pred_dict={m:np.load(os.path.join(PRED_DIR,f'{m}_test_preds.npy')) for m in available}
    weighted_weights=None
    
    if RUN_WEIGHTED_ENSEMBLE:
        try:
            row, weighted_weights, pred = run_weighted_soft_voting(available, metrics_df); ensemble_rows.append(row); pred_dict['weighted_soft_voting_ensemble']=pred
        except Exception as e: print('Weighted ensemble failed:',e)
    
    try:
        top_rows=run_selective_topk(metrics_df[metrics_df.model.isin(available)]); ensemble_rows += top_rows
        for k in SELECTIVE_TOP_K_LIST:
            fp=os.path.join(PRED_DIR,f'selective_weighted_soft_voting_top_{k}_test_preds.npy')
            if os.path.exists(fp): pred_dict[f'selective_weighted_soft_voting_top_{k}']=np.load(fp)
    except Exception as e: print('Top-K ensemble failed:',e)
    
    if RUN_STACKING_ENSEMBLE:
        try:
            row,pred=run_stacking_ensemble(available); ensemble_rows.append(row); pred_dict['stacking_logistic_regression_ensemble']=pred
        except Exception as e: print('Stacking failed:',e)
    
    if ensemble_rows:
        ens_df=pd.DataFrame(ensemble_rows); ens_df.to_csv(os.path.join(STAT_DIR,'all_ensemble_train_val_test_results.csv'),index=False)
        pd.concat([metrics_df.assign(method_type='individual_cnn'), ens_df.assign(method_type='ensemble')], ignore_index=True).to_csv(os.path.join(STAT_DIR,'final_all_methods_train_val_test_results.csv'),index=False)
    
    if pred_dict:
        mdf=mcnemar_pairwise(pred_dict,test_labels); mdf.to_csv(os.path.join(STAT_DIR,'mcnemar_pairwise_all_methods.csv'),index=False)
    
    if RUN_ENSEMBLE_GRADCAM and weighted_weights is not None:
        wfp=os.path.join(PRED_DIR,'weighted_soft_voting_ensemble_test_preds.npy')
        if os.path.exists(wfp): generate_ensemble_gradcam('weighted_soft_voting_ensemble', available, weighted_weights, np.load(wfp), test_paths, test_labels, average_mode=False, n=GRADCAM_N_IMAGES)
        sfp=os.path.join(PRED_DIR,'stacking_logistic_regression_ensemble_test_preds.npy')
        if os.path.exists(sfp): generate_ensemble_gradcam('stacking_logistic_regression_ensemble', available, np.ones(len(available)), np.load(sfp), test_paths, test_labels, average_mode=True, n=GRADCAM_N_IMAGES)
    
    print('PIPELINE COMPLETED:', datetime.now())
    print('Results:', OUTPUT_DIR)

if __name__ == '__main__':
    main()

2026-06-08 19:40:50.849031: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780947651.055083      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780947651.113125      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780947651.579535      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780947651.579580      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780947651.579583      22 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
✅ Standard float32 precision enforced for P100 architecture
✅ XLA compiler enabled
Initial RAM: used=1.26GB | available=31.93GB

✅ Downloaded to: /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20
Train: /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/train
Val:   /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/val
Test:  /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/test
Classes: ['Glioma Tumor', 'Meningioma Tumor', 'Normal Brain', 'Pituitary tumor']
Meningioma index: 1 Meningioma Tumor
Train=1394 | Val=196 | Test=403

PIPELINE STARTED: 2026-06-08 19:41:09.964444

========== 5-FOLD CV: mobilenetv2 ==========
Fold 1/5 - mobilenetv2
Meningioma oversampling: added 138


I0000 00:00:1780947670.079227      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8


I0000 00:00:1780947677.083890      64 service.cc:152] XLA service 0x7d76ac005690 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780947677.083931      64 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780947677.116913      64 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780947677.275886      64 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - accuracy: 0.6478 - loss: 1.1246
Epoch 1: val_accuracy improved from None to 0.82653, saving model to /kaggle/working/brain_tumor_results_advanced_bigv10.1 all/models/efficientnetb0_final_best.weights.h5

Epoch 1: finished saving model to /kaggle/working/brain_tumor_results_advanced_bigv10.1 all/models/efficientnetb0_final_best.weights.h5
98/98 ━━━━━━━━━━━━━━━━━━━━ 72s 357ms/step - accuracy: 0.6592 - loss: 1.1056 - val_accuracy: 0.8265 - val_loss: 0.7465
Epoch 2/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6548 - loss: 1.0913
Epoch 2: val_accuracy did not improve from 0.82653
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.6847 - loss: 1.0201 - val_accuracy: 0.8112 - val_loss: 0.7606
Epoch 3/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6821 - loss: 1.0126
Epoch 3: val_accuracy did not improve from 0.82653
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.7128 - loss: 0.9581 - val_accuracy: 0.8061 - val_loss: 0.

2026-06-08 20:52:15.481358: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-08 20:52:15.708124: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


194/196 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6410 - loss: 1.1375

2026-06-08 20:52:57.415938: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-08 20:52:57.643345: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.6413 - loss: 1.1371
Epoch 1: val_accuracy improved from None to 0.79082, saving model to /kaggle/working/brain_tumor_results_advanced_bigv10.1 all/models/efficientnetb4_final_best.weights.h5

Epoch 1: finished saving model to /kaggle/working/brain_tumor_results_advanced_bigv10.1 all/models/efficientnetb4_final_best.weights.h5
196/196 ━━━━━━━━━━━━━━━━━━━━ 133s 361ms/step - accuracy: 0.6707 - loss: 1.0999 - val_accuracy: 0.7908 - val_loss: 0.8667
Epoch 2/10
195/196 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.6647 - loss: 1.0823
Epoch 2: val_accuracy did not improve from 0.79082
196/196 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6841 - loss: 1.0459 - val_accuracy: 0.7602 - val_loss: 0.8757
Epoch 3/10
195/196 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6846 - loss: 1.0997
Epoch 3: val_accuracy did not improve from 0.79082
196/196 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - accuracy: 0.7147 - loss: 1.0204 - val_accuracy: 0.7653

In [2]:
import shutil

OUTPUT_DIR = "/kaggle/working/brain_tumor_results_advanced_bigv10.1 all"

shutil.make_archive(
    OUTPUT_DIR,
    "zip",
    OUTPUT_DIR
)

print("Saved:", OUTPUT_DIR + ".zip")

Saved: /kaggle/working/brain_tumor_results_advanced_bigv10.1 all.zip


In [3]:
import os

print(os.path.exists("//kaggle/working/brain_tumor_results_advanced_bigv10.1 all"))
print(os.path.getsize("/kaggle/working/brain_tumor_results_advanced_bigv10.1 all") / 1024**2, "MB")

True
0.00390625 MB


In [4]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import gc, math, json, time, shutil, random, warnings
from datetime import datetime
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import psutil
import joblib
import kagglehub

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import StratifiedKFold
import sklearn.metrics as skm
from sklearn.preprocessing import label_binarize
from sklearn.linear_model import LogisticRegression

try:
    from statsmodels.stats.contingency_tables import mcnemar
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    from statsmodels.stats.contingency_tables import mcnemar

from tensorflow.keras.applications import EfficientNetB0, EfficientNetB4, MobileNetV2, MobileNetV3Large, ResNet50V2, VGG16
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mnv2
from tensorflow.keras.applications.resnet_v2 import preprocess_input as preprocess_resnetv2
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16

In [5]:
# ============================================================
# TOP-2 / TOP-3 / TOP-4 ENSEMBLE GRADCAM++ (NO RETRAINING)
# Uses saved weights + saved ensemble outputs + test images
# ============================================================

import os, math, gc, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ----------------------------
# 1) PATHS
# ----------------------------
ROOT = "/kaggle/input/datasets/nahidulislamnihal/brain-tumor-results-advanced-bigv10-1-c26"
DATASET_ROOT = "/kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/test"  # change if needed

MODEL_DIR = os.path.join(ROOT, "models")
PRED_DIR = os.path.join(ROOT, "predictions")
STAT_DIR = os.path.join(ROOT, "statistics")
GRADCAM_DIR = "/kaggle/working/gradcam"
os.makedirs(GRADCAM_DIR, exist_ok=True)

for d in [GRADCAM_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
MODELS_TO_RUN = ['mobilenetv2','mobilenetv3','resnet50v2','efficientnetb0','efficientnetb4','vgg16']
IMG_SIZE_DICT = {m:(224,224) for m in MODELS_TO_RUN}
BATCH_SIZE_DICT = {'mobilenetv2':16,'mobilenetv3':16,'resnet50v2':16,'efficientnetb0':16,'vgg16':16,'efficientnetb4':8}

DROPOUT_RATE = 0.4
DENSE_UNITS = 256
WEIGHT_DECAY = 1e-4
GRADCAM_ALPHA = 0.30
GRADCAM_THRESHOLD_PERCENTILE = 85
USE_GRADCAM_PLUS_PLUS = True
GRADCAM_N_IMAGES = 8

# ----------------------------
# 2) DATASET DISCOVERY
# ----------------------------
def get_class_names(test_dir):
    parent = os.path.dirname(test_dir)
    fp = os.path.join(parent, "class_names.txt")
    if os.path.exists(fp):
        return [x.strip() for x in open(fp, encoding="utf-8") if x.strip()]
    return sorted([d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))])

def list_paths_and_labels(split_dir, class_names):
    paths, labels = [], []
    for i, c in enumerate(class_names):
        cd = os.path.join(split_dir, c)
        if not os.path.isdir(cd):
            continue
        for p in Path(cd).rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                paths.append(str(p))
                labels.append(i)
    return np.array(paths), np.array(labels, dtype=np.int32)

assert os.path.isdir(DATASET_ROOT), f"Test folder not found: {DATASET_ROOT}"
CLASS_NAMES = get_class_names(DATASET_ROOT)
NUM_CLASSES = len(CLASS_NAMES)

test_paths, test_labels = list_paths_and_labels(DATASET_ROOT, CLASS_NAMES)

print("Classes:", CLASS_NAMES)
print("Test images:", len(test_paths))

# ----------------------------
# 3) FIXED IMAGE SELECTION (same logic as original)
# ----------------------------
def select_balanced_indices(labels, n=8):
    rng = np.random.default_rng(SEED)
    idx = []
    per = max(1, math.ceil(n / NUM_CLASSES))
    labels = np.array(labels)

    for c in range(NUM_CLASSES):
        ci = np.where(labels == c)[0]
        if len(ci):
            idx += rng.choice(ci, size=min(per, len(ci)), replace=False).tolist()

    if len(idx) > n:
        idx = rng.choice(idx, size=n, replace=False).tolist()
    return idx

fixed_indices = select_balanced_indices(test_labels, GRADCAM_N_IMAGES)

print("\nFixed indices used for all figures:")
print(fixed_indices)
print("\nSelected image paths:")
for i in fixed_indices:
    print(i, test_paths[i])

SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

pd.DataFrame({
    "index": fixed_indices,
    "image_path": [test_paths[i] for i in fixed_indices],
    "true_label": [CLASS_NAMES[int(test_labels[i])] for i in fixed_indices]
}).to_csv(
    os.path.join(SAVE_DIR, "selected_gradcam_images.csv"),
    index=False
)

print("Saved successfully.")

# ----------------------------
# 4) PREPROCESSING / MODEL BUILDING
# ----------------------------
from tensorflow.keras.applications import EfficientNetB0, EfficientNetB4, MobileNetV2, MobileNetV3Large, ResNet50V2, VGG16
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mnv2
from tensorflow.keras.applications.resnet_v2 import preprocess_input as preprocess_resnetv2
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16

def identity_preprocess(x):
    return x

PREPROCESS_DICT = {
    'mobilenetv2': preprocess_mnv2,
    'mobilenetv3': identity_preprocess,
    'resnet50v2': preprocess_resnetv2,
    'efficientnetb0': preprocess_eff,
    'efficientnetb4': preprocess_eff,
    'vgg16': preprocess_vgg16
}

def get_batch_size(m):
    return BATCH_SIZE_DICT.get(m, 16)

def get_backbone(model_name, inp, input_shape):
    try:
        if model_name == 'mobilenetv2':
            return MobileNetV2(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
        if model_name == 'mobilenetv3':
            return MobileNetV3Large(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape, include_preprocessing=True)
        if model_name == 'resnet50v2':
            return ResNet50V2(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
        if model_name == 'efficientnetb0':
            return EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
        if model_name == 'efficientnetb4':
            return EfficientNetB4(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
        if model_name == 'vgg16':
            return VGG16(include_top=False, weights='imagenet', input_tensor=inp, input_shape=input_shape)
    except Exception as e:
        print(f"ImageNet weights unavailable for {model_name}: {e}")

    if model_name == 'mobilenetv2':
        return MobileNetV2(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    if model_name == 'mobilenetv3':
        return MobileNetV3Large(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape, include_preprocessing=True)
    if model_name == 'resnet50v2':
        return ResNet50V2(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    if model_name == 'efficientnetb0':
        return EfficientNetB0(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    if model_name == 'efficientnetb4':
        return EfficientNetB4(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    if model_name == 'vgg16':
        return VGG16(include_top=False, weights=None, input_tensor=inp, input_shape=input_shape)
    raise ValueError(model_name)

def build_model(model_name):
    input_shape = IMG_SIZE_DICT[model_name] + (3,)
    inp = keras.Input(shape=input_shape)
    base = get_backbone(model_name, inp, input_shape)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(DENSE_UNITS, activation='relu', kernel_regularizer=keras.regularizers.l2(WEIGHT_DECAY))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    return keras.Model(inp, out, name=model_name), base

def find_last_conv_layer(model):
    convs = (layers.Conv2D, layers.DepthwiseConv2D, layers.SeparableConv2D)
    for l in reversed(model.layers):
        if isinstance(l, convs):
            return l.name
    for l in reversed(model.layers):
        try:
            if len(l.output.shape) == 4:
                return l.name
        except Exception:
            pass
    return None

def normalize_heatmap(h):
    h = np.nan_to_num(h)
    h = np.maximum(h, 0)
    mx = np.max(h)
    return (h / mx).astype(np.float32) if mx > 0 else h.astype(np.float32)

def make_gradcam_heatmap(model, img_batch, last_conv, pred_idx):
    gm = keras.Model(model.inputs, [model.get_layer(last_conv).output, model.output])
    with tf.GradientTape() as tape:
        conv, pred = gm(img_batch, training=False)
        loss = pred[:, pred_idx]
    grads = tape.gradient(loss, conv)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.reduce_sum(conv[0] * weights, axis=-1)
    return normalize_heatmap(tf.nn.relu(cam).numpy())

def make_gradcampp_heatmap(model, img_batch, last_conv, pred_idx):
    gm = keras.Model(model.inputs, [model.get_layer(last_conv).output, model.output])
    with tf.GradientTape() as tape:
        conv, pred = gm(img_batch, training=False)
        loss = pred[:, pred_idx]
    grads = tape.gradient(loss, conv)[0].numpy()
    conv = conv[0].numpy()
    first = grads
    second = grads ** 2
    third = grads ** 3
    gsum = np.sum(conv, axis=(0, 1), keepdims=True)
    denom = 2 * second + third * gsum
    denom = np.where(denom != 0, denom, 1e-8)
    alphas = second / denom
    weights = np.sum(alphas * np.maximum(first, 0), axis=(0, 1))
    cam = np.sum(weights * conv, axis=-1)
    return normalize_heatmap(cam)

def focus_heatmap(h, percentile=85):
    h = normalize_heatmap(h)
    if np.max(h) <= 0:
        return h
    thr = np.percentile(h, percentile)
    mask = (h >= thr).astype(np.uint8)
    try:
        n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        if n > 1:
            largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
            mask = (lab == largest).astype(np.uint8)
    except Exception:
        pass
    out = h * mask
    if out.shape[0] >= 5 and out.shape[1] >= 5:
        out = cv2.GaussianBlur(out, (5, 5), 0)
    return normalize_heatmap(out)

def overlay_heatmap(img, h, alpha=0.30):
    img = img.astype(np.uint8)
    hh, ww = img.shape[:2]
    h = cv2.resize(normalize_heatmap(h), (ww, hh))
    h = np.uint8(255 * h)
    col = cv2.applyColorMap(h, cv2.COLORMAP_JET)
    col = cv2.cvtColor(col, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img, 1 - alpha, col, alpha, 0)

def load_image_for_cam(path, model_name):
    size = IMG_SIZE_DICT[model_name]
    pre = PREPROCESS_DICT[model_name]
    raw = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    raw.set_shape([None, None, 3])
    raw = tf.cast(raw, tf.float32)
    raw = tf.image.resize_with_pad(raw, size[0], size[1])
    raw_np = tf.cast(raw, tf.uint8).numpy()
    batch = pre(tf.expand_dims(raw, 0))
    return raw_np, batch

# ----------------------------
# 5) ENSEMBLE GRADCAM++ GENERATOR
# ----------------------------
def load_topk_weights(topk):
    fp = os.path.join(STAT_DIR, f"selective_weighted_soft_voting_top_{topk}_weights.csv")
    df = pd.read_csv(fp)
    if "weight_from_validation_macro_f1" in df.columns:
        wcol = "weight_from_validation_macro_f1"
    elif "weight" in df.columns:
        wcol = "weight"
    else:
        raise ValueError(f"Weight column not found in {fp}")
    models = df["model"].tolist()
    weights = df[wcol].astype(np.float32).values
    weights = weights / (weights.sum() + 1e-8)
    return models, weights

def load_topk_preds(topk):
    fp = os.path.join(PRED_DIR, f"selective_weighted_soft_voting_top_{topk}_test_preds.npy")
    if os.path.exists(fp):
        return np.load(fp)
    fp2 = os.path.join(PRED_DIR, f"selective_weighted_soft_voting_top_{topk}_test_probs.npy")
    if os.path.exists(fp2):
        return np.argmax(np.load(fp2), axis=1)
    raise FileNotFoundError(f"Top-{topk} preds/probs not found in {PRED_DIR}")

def generate_ensemble_gradcam_fixed(ensemble_name, model_names, model_weights, target_preds, paths, labels_array, fixed_indices):
    size = IMG_SIZE_DICT[model_names[0]]
    fused = {i: np.zeros(size, dtype=np.float32) for i in fixed_indices}
    raw_imgs = {}

    for m, w in zip(model_names, model_weights):
        weight_file = os.path.join(MODEL_DIR, f"{m}_final_best.weights.h5")
        if not os.path.exists(weight_file):
            raise FileNotFoundError(weight_file)

        model, _ = build_model(m)
        model.load_weights(weight_file)
        last = find_last_conv_layer(model)
        if not last:
            raise ValueError(f"No conv layer found for {m}")

        print(f"Loaded {m} | last conv layer: {last}")

        for idx in fixed_indices:
            raw, b = load_image_for_cam(paths[idx], m)
            raw_imgs.setdefault(idx, raw)
            target = int(target_preds[idx])

            if USE_GRADCAM_PLUS_PLUS:
                h = make_gradcampp_heatmap(model, b, last, target)
            else:
                h = make_gradcam_heatmap(model, b, last, target)

            fused[idx] += float(w) * cv2.resize(normalize_heatmap(h), (size[1], size[0]))

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    cols = 4
    rows = math.ceil(len(fixed_indices) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax_i, idx in enumerate(fixed_indices):
        true = int(labels_array[idx])
        pred = int(target_preds[idx])
        h = focus_heatmap(fused[idx], GRADCAM_THRESHOLD_PERCENTILE)
        over = overlay_heatmap(raw_imgs[idx], h, GRADCAM_ALPHA)
        axes[ax_i].imshow(over)
        axes[ax_i].set_title(
            f"True: {CLASS_NAMES[true]}\nEnsemble Pred: {CLASS_NAMES[pred]}",
            color='green' if pred == true else 'red'
        )
        axes[ax_i].axis("off")

    for j in range(len(fixed_indices), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    out_fp = os.path.join(GRADCAM_DIR, f"{ensemble_name}_focused_ensemble_gradcampp.png")
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved:", out_fp)

# ----------------------------
# 6) RUN TOP-2 / TOP-3 / TOP-4
# ----------------------------
for topk in [2, 3, 4]:
    print("\n" + "="*70)
    print(f"Generating Top-{topk} ensemble Grad-CAM++")
    model_names, model_weights = load_topk_weights(topk)
    target_preds = load_topk_preds(topk)
    generate_ensemble_gradcam_fixed(
        ensemble_name=f"selective_weighted_soft_voting_top_{topk}",
        model_names=model_names,
        model_weights=model_weights,
        target_preds=target_preds,
        paths=test_paths,
        labels_array=test_labels,
        fixed_indices=fixed_indices
    )

print("\nDone.")

Classes: ['Glioma Tumor', 'Meningioma Tumor', 'Normal Brain', 'Pituitary tumor']
Test images: 403

Fixed indices used for all figures:
[8, 78, 144, 200, 271, 209, 311, 355]

Selected image paths:
8 /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/test/Glioma Tumor/ac1853404f3a_G-49.jpg
78 /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/test/Glioma Tumor/ee6517ecc56b_G-81.jpg
144 /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/test/Meningioma Tumor/78f62cdc0cab_M-345.jpg
200 /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/test/Meningioma Tumor/48ad5a0b1b22_M-377.jpg
271 /kaggle/input/datasets/nahidulislamnihal/clean-brain-tumor-dataset-v0870-10-20/Clean_Brain_Tumor_Dataset_v08(70-10-20)/test/Normal Brain/3f7c